# A*: Aerial Refueling Route Optimization
### Nathan DiMatteo, Thomas Gallegos, Kyle Squires

In [9]:
%pip install plotly nbformat

from collections import deque
import heapq
from math import inf
import numpy as np
import plotly.graph_objects as go


Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 25.2 -> 26.3
[notice] To update, run: python.exe -m pip install --upgrade pip


### 1. Path reconstruction and A* functions

In [10]:
def reconstruct_path(parent, goal):
    if goal not in parent:
        return None
    path, current = [], goal
    while current is not None:
        path.append(current)
        current = parent[current]
    return list(reversed(path))


def astar(graph,start,goal,h):
    frontier = [(h[start],0,start,None)] #creates queue to be explored each item is (f, g, current node, came from node) 
    best,parent,expanded = {start:0},{},[] #holds the best cost and path to each node

    # loop through the frontier, removing node with the smallest f value first  
    while frontier:
        _,g,node,predecessor = heapq.heappop(frontier)

        # continue only if current cost is the best cost to get to this node 
        if g != best.get(node,inf):
            continue
        # record parent
        parent[node]=predecessor; expanded.append(node)
        
        #check if at the end of the path (goal)
        if node == goal:
            break
        for neighbor,step in graph[node]:
            new_g = g+step
            if new_g < best.get(neighbor,inf):
                best[neighbor]=new_g
                heapq.heappush(frontier,(new_g+h[neighbor],new_g,neighbor,node))
    path = reconstruct_path(parent,goal)
    return {"algorithm":"A*","path":path,"cost":best.get(goal,inf),"expanded":expanded}

### 2. Graph coordinates and connections

In [11]:
# ---------------------------------------------------------------- waypoints
coords = {
    "Base":           (5, 5, 4),
    "Low West":       (18, 30, 8),
    "South Valley":   (45, 12, 6),
    "Ridge Climb":    (33, 42, 22),
    "Far East":       (98, 10, 10),
    "High East":      (98, 10, 30),
    "Summit Pass":    (50, 50, 27),
    "North Corridor": (48, 78, 18),
    "East Ridge":     (66, 44, 21),
    "Jet Stream":     (66, 66, 25),
    "Radar Gap":      (96, 48, 14),
    "Tanker Track":   (88, 84, 24),
    "Tanker":         (95, 92, 25),
}
start, goal = "Base", "Tanker"

# Possible flights between waypoints (one-way). Costs are NOT set here;
# they are calculated below, and blocked connections are dropped.
connections = {
    "Base":           ["Low West", "South Valley", "Ridge Climb", "Far East", "High East"],
    "Low West":       ["Ridge Climb", "South Valley"],
    "South Valley":   ["East Ridge", "Summit Pass"],
    "Ridge Climb":    ["Summit Pass", "North Corridor"],
    "Far East":       ["Radar Gap"],
    "High East":      ["Radar Gap", "Tanker Track"],
    "Summit Pass":    ["Jet Stream", "East Ridge"],
    "North Corridor": ["Tanker Track"],
    "East Ridge":     ["Jet Stream", "Radar Gap"],
    "Jet Stream":     ["Tanker Track", "Tanker"],
    "Radar Gap":      ["Tanker Track"],
    "Tanker Track":   ["Tanker"],
    "Tanker":         [],
}


### 3. Other Helper Functions 
- Calculate heuristic and actual path costs
    - Edge costs are calculated every run from `coords` and the hazard settings:
        - cost = distance/5 + climb/4 + WIND_PENALTY * wind_fraction * distance/5
- Connections that enter restricted airspace or a radar dome, or pass within TERRAIN_CLEARANCE of the mountain, are removed from the graph.


- Create graph A* algoritm will use

In [12]:

# ---------------------------------------------------------------- hazards
MOUNTAIN = dict(cx=50, cy=50, peak=20, sigma=12)
RESTRICTED_ZONES = [                                  # boxes: (min, max) per axis
    dict(x=(20, 38), y=(60, 100), z=(0, 35)),
    dict(x=(60, 90), y=(20, 30),  z=(0, 35)),
]
WIND = dict(cx=70, cy=70, r=16, z=(22, 30))           # cylinder

# ---------------------------------------------------------------- cost settings
DIST_SCALE        = 5      # One unit of fuel can fly (5) units of distance
CLIMB_SCALE       = 4      # One unit of fuel can fly (4) units of climb
WIND_PENALTY      = 6      # extra multiplier on distance cost while inside wind
TERRAIN_CLEARANCE = 3      # kft required above the mountain
SAMPLES           = 2000    # points checked along each edge


#Creates reasonable heuristic using straight line distance + minimum climb from current point. 
# ensures that the heuristic is always less than or equal to the actual cost
def heuristic(node):
    """Optimistic estimate of the cost from node to the goal."""
    x, y, z = coords[node]
    gx, gy, gz = coords[goal]
    straight_line = np.sqrt((gx - x)**2 + (gy - y)**2 + (gz - z)**2)
    climb_needed = max(0, gz - z)
    return straight_line / DIST_SCALE + climb_needed / CLIMB_SCALE

heuristic_reasonable = {node: heuristic(node) for node in coords}


# Create terrain
def terrain(x, y):
    m = MOUNTAIN
    return m["peak"] * np.exp(-((x - m["cx"])**2 + (y - m["cy"])**2) / (2 * m["sigma"]**2))


# check if point x,y,z is within the box, b, range
def in_box(x, y, z, b):
    return (b["x"][0] <= x) & (x <= b["x"][1]) & (b["y"][0] <= y) & (y <= b["y"][1]) \
         & (b["z"][0] <= z) & (z <= b["z"][1])

# check if point x, y, z, is in wind range
def in_wind(x, y, z, w=WIND):
    return ((x - w["cx"])**2 + (y - w["cy"])**2 <= w["r"]**2) & (w["z"][0] <= z) & (z <= w["z"][1])

# cut each edge into many points, check if each point is in any box and how many points are in wind zone
def check_edge(a, b):
    """Return (cost, None) for a legal edge, or (None, reason) if it is blocked."""
    a, b = np.asarray(a, float), np.asarray(b, float)
    t = np.linspace(0, 1, SAMPLES)[:, None]
    x, y, z = (a + t * (b - a)).T

    for n, box in enumerate(RESTRICTED_ZONES, 1):
        if in_box(x, y, z, box).any():
            return None, f"restricted airspace #{n}"
    if (z < terrain(x, y) + TERRAIN_CLEARANCE).any():
        return None, "terrain clearance"

    dist = np.linalg.norm(b - a)
    climb = max(0.0, b[2] - a[2])
    wind_frac = in_wind(x, y, z).mean()
    cost = dist / DIST_SCALE + climb / CLIMB_SCALE + WIND_PENALTY * wind_frac * dist / DIST_SCALE
    return float(cost), None

#build graph that A* will use 
def build_graph(verbose=True):
    """Build {node: [(neighbor, cost), ...]} from coords, connections and hazards."""
    #start with empty list, create graph list for each node adn a blocked list to seperate bad entries
    graph, blocked = {n: [] for n in coords}, []
    # loop through each node in the graph
    for node, nbrs in connections.items():
        #loop thruogh each connection in each node and add cost or remove if cost is 0
        for nbr in nbrs:
            cost, reason = check_edge(coords[node], coords[nbr])
            if cost is None:
                blocked.append((node, nbr, reason))
            else:
                graph[node].append((nbr, cost))
    #print each point and cost
    if verbose:
        for node, nbrs in graph.items():
            print(f"{node:15s} " + ", ".join(f"{n} ({c:.2f})" for n, c in nbrs))
        for node, nbr, reason in blocked:
            print(f"BLOCKED  {node} -> {nbr}: {reason}")
    return graph, blocked


refuel_graph, blocked_edges = build_graph()

Base            Low West (6.69), South Valley (8.63), Ridge Climb (14.45), Far East (20.17), High East (25.84)
Low West        Ridge Climb (8.25), South Valley (6.50)
South Valley    East Ridge (11.97), Summit Pass (13.99)
Ridge Climb     Summit Pass (5.14), North Corridor (7.84)
Far East        Radar Gap (8.65)
High East       Radar Gap (8.26), Tanker Track (14.98)
Summit Pass     Jet Stream (17.00), East Ridge (3.62)
North Corridor  Tanker Track (15.81)
East Ridge      Jet Stream (19.49), Radar Gap (6.21)
Jet Stream      Tanker Track (31.65), Tanker (33.75)
Radar Gap       Tanker Track (10.14)
Tanker Track    Tanker (2.39)
Tanker          


Ploting helper functions

In [13]:

# ---------------------------------------------------------------- plot helpers
def solid(color):
    """Single-color colorscale for go.Surface."""
    return [[0, color], [1, color]]


def ring(cx, cy, r, z, n=60):
    t = np.linspace(0, 2 * np.pi, n)
    return cx + r * np.cos(t), cy + r * np.sin(t), np.full(n, z)


def join(segments):
    """Concatenate line segments with None breaks for one Scatter3d trace."""
    xs, ys, zs = [], [], []
    for x, y, z in segments:
        xs += list(x) + [None]; ys += list(y) + [None]; zs += list(z) + [None]
    return xs, ys, zs


# ---------------------------------------------------------------- traces
def mountain_trace():
    g = np.linspace(0, 100, 100)
    X, Y = np.meshgrid(g, g)
    return go.Surface(
        x=X, y=Y, z=terrain(X, Y), name="Mountain (3k ft clearance)",
        colorscale=[[0, "#e8e2c8"], [0.3, "#b5a67a"], [0.7, "#8a6d45"], [1, "#f4f4f4"]],
        showscale=False, showlegend=True, opacity=1.0,
        hovertemplate="Terrain: %{z:.1f} kft<extra></extra>",
        lighting=dict(ambient=0.6, diffuse=0.8, roughness=0.9),
    )


def restricted_traces():
    traces = []
    for n, b in enumerate(RESTRICTED_ZONES):
        (x0, x1), (y0, y1), (z0, z1) = b["x"], b["y"], b["z"]
        vx = [x0, x1, x1, x0, x0, x1, x1, x0]
        vy = [y0, y0, y1, y1, y0, y0, y1, y1]
        vz = [z0, z0, z0, z0, z1, z1, z1, z1]
        i = [0, 0, 4, 4, 0, 0, 1, 1, 2, 2, 3, 3]
        j = [1, 2, 5, 6, 1, 5, 2, 6, 3, 7, 0, 4]
        k = [2, 3, 6, 7, 5, 4, 6, 5, 7, 6, 4, 7]
        traces.append(go.Mesh3d(x=vx, y=vy, z=vz, i=i, j=j, k=k, color="red", opacity=0.25,
                                name="Restricted airspace", legendgroup="restricted",
                                showlegend=(n == 0), hoverinfo="name", flatshading=True))
        edges = [(0, 1), (1, 2), (2, 3), (3, 0), (4, 5), (5, 6), (6, 7), (7, 4),
                 (0, 4), (1, 5), (2, 6), (3, 7)]
        ex, ey, ez = join([([vx[a], vx[c]], [vy[a], vy[c]], [vz[a], vz[c]]) for a, c in edges])
        traces.append(go.Scatter3d(x=ex, y=ey, z=ez, mode="lines", line=dict(color="red", width=5),
                                   legendgroup="restricted", showlegend=False, hoverinfo="skip"))
    return traces



def wind_traces():
    w = WIND
    th, zz = np.meshgrid(np.linspace(0, 2 * np.pi, 50), np.linspace(*w["z"], 2))
    X = w["cx"] + w["r"] * np.cos(th)
    Y = w["cy"] + w["r"] * np.sin(th)
    cyl = go.Surface(x=X, y=Y, z=zz, surfacecolor=np.zeros_like(zz), colorscale=solid("deepskyblue"),
                     showscale=False, opacity=0.25, name="High-wind zone (cost penalty)",
                     legendgroup="wind", showlegend=True, hoverinfo="name")
    ex, ey, ez = join([ring(w["cx"], w["cy"], w["r"], w["z"][0]),
                       ring(w["cx"], w["cy"], w["r"], w["z"][1])])
    outline = go.Scatter3d(x=ex, y=ey, z=ez, mode="lines", line=dict(color="deepskyblue", width=4),
                           legendgroup="wind", showlegend=False, hoverinfo="skip")
    return [cyl, outline]


def edge_traces():
    segs, mx, my, mz, labels, short = [], [], [], [], [], []
    for node, nbrs in refuel_graph.items():
        for nbr, c in nbrs:
            (x0, y0, z0), (x1, y1, z1) = coords[node], coords[nbr]
            segs.append(([x0, x1], [y0, y1], [z0, z1]))
            mx.append((x0 + x1) / 2); my.append((y0 + y1) / 2); mz.append((z0 + z1) / 2)
            labels.append(f"{node} → {nbr}<br>cost {c:.2f}")
            short.append(f"{c:.1f}")
    ex, ey, ez = join(segs)
    lines = go.Scatter3d(x=ex, y=ey, z=ez, mode="lines", line=dict(color="#333333", width=4),
                         name="Edges", legendgroup="edges", hoverinfo="skip")
    costs = go.Scatter3d(x=mx, y=my, z=mz, mode="markers+text", text=short, hovertext=labels,
                         hoverinfo="text", textposition="top center",
                         textfont=dict(size=12, color="black"),
                         marker=dict(size=3, color="#333333"),
                         name="Edge costs", legendgroup="edges", showlegend=False)
    return [lines, costs]


def blocked_edge_trace():
    """Removed connections, dashed; hidden until clicked in the legend."""
    if not blocked_edges:
        return []
    ex, ey, ez = join([([coords[a][0], coords[b][0]], [coords[a][1], coords[b][1]],
                        [coords[a][2], coords[b][2]]) for a, b, _ in blocked_edges])
    return [go.Scatter3d(x=ex, y=ey, z=ez, mode="lines",
                         line=dict(color="red", width=3, dash="dash"),
                         name="Blocked connections", visible="legendonly", hoverinfo="skip")]


def node_trace():
    names = list(coords)
    xs, ys, zs = zip(*coords.values())
    colors = ["green" if n == start else "crimson" if n == goal else "navy" for n in names]
    sizes = [11 if n in (start, goal) else 7 for n in names]
    hover = [f"<b>{n}</b><br>Altitude: {z} kft<br>Terrain clearance: {z - terrain(x, y):.1f} kft"
             for n, (x, y, z) in coords.items()]
    return go.Scatter3d(x=xs, y=ys, z=zs, mode="markers+text", text=names,
                        textposition="top center", textfont=dict(size=13, color="black"),
                        marker=dict(size=sizes, color=colors, line=dict(color="white", width=1)),
                        hovertext=hover, hoverinfo="text", name="Waypoints")


# ---------------------------------------------------------------- figure
def build_figure():
    fig = go.Figure([mountain_trace(), *restricted_traces(),
                     *wind_traces(), *edge_traces(), *blocked_edge_trace(), node_trace()])
    fig.update_layout(
        title="3D Aerial Refueling Route Graph  (start: Base → goal: Tanker)",
        scene=dict(
            xaxis=dict(title="East (x)", range=[0, 100]),
            yaxis=dict(title="North (y)", range=[0, 100]),
            zaxis=dict(title="Altitude (kft)", range=[0, 35]),
            aspectmode="manual", aspectratio=dict(x=1, y=1, z=0.6),
            camera=dict(eye=dict(x=-1.4, y=-1.6, z=0.9)),
        ),
        legend=dict(x=0.01, y=0.99, bgcolor="rgba(255,255,255,0.85)"),
        margin=dict(l=0, r=0, t=40, b=0), height=750,
    )
    return fig


if __name__ == "__main__":
    fig = build_figure()
    fig.write_html("refuel_graph_3d.html")   # standalone file, opens in any browser
    fig.show()

### 4. A* (A-Star) Algorithm implementation

In [14]:
for r in [
    astar(refuel_graph,"Base","Tanker",heuristic_reasonable)
]:
    print(f"\n{r['algorithm']}")
    print("Path:", " -> ".join(r["path"]))
    print("Fuel cost:", r["cost"])
    print("Expansion order:", r["expanded"])


A*
Path: Base -> South Valley -> East Ridge -> Radar Gap -> Tanker Track -> Tanker
Fuel cost: 39.34363720356724
Expansion order: ['Base', 'Low West', 'Ridge Climb', 'Summit Pass', 'South Valley', 'East Ridge', 'North Corridor', 'Radar Gap', 'Tanker Track', 'Tanker']


### 5. Correctness Checks
In the first correctness check our astar is compared with dijkstras, which uses a 0 heuristic, therefore finding the cheapest path. This ensures our algorithm is working correctly.

In the second test it checks the admissibility of the heuristic to make sure that it isn't overestimating. 


In [15]:
# Check 1: A* finds the same cost as Dijkstra's algorithm
zero_heuristic = {node: 0 for node in coords}
astar_result = astar(refuel_graph, start, goal, heuristic_reasonable)
dijkstra_result = astar(refuel_graph, start, goal, zero_heuristic)

assert np.isclose(astar_result["cost"], dijkstra_result["cost"]), \
    "A* cost does not match Dijkstra's cost"

# Check 2: The heuristic never exceeds the true remaining cost
for node in coords:
    true_cost = astar(refuel_graph, node, goal, zero_heuristic)["cost"]
    if np.isfinite(true_cost):
        assert heuristic_reasonable[node] <= true_cost + 1e-9, \
            f"Heuristic overestimates from {node}"

print("Both correctness checks passed.")

Both correctness checks passed.


## Documentation Statement: 
I used Claude to help build the 3D graph environment and visualization for this project. I defined the scenario and requirements: an aircraft routing to a refueling tanker through restricted airspace, a mountain, high-wind areas and radar zones, modeled on the format of our 2D course graph. Given this, Claude generated the initial waypoint graph, the hazard geometry, the visualization code, the edge-cost calculations (check_edge, build_graph and the zone-check helpers), and the straight-line-plus-climb heuristic. It also checked that the heuristic never overestimates the true cost.

I directed revisions over many rounds. I had the visualization moved from matplotlib to Plotly for readability, and had the hard-coded edge costs replaced with costs calculated from waypoint coordinates and hazard settings each run. I modified the environment myself: I added the Far East and High East waypoints and their connections, added a second restricted zone, removed the radar zone, resized the wind zone and the first restricted zone, and increased the sampling resolution. I also asked Claude to explain each function (the cost formula, edge sampling, in_box, build_graph, the plotting traces) until I understood it, and wrote my own code comments based on that understanding. While I did not directly change the functions, I consistently iterated with the AI model in order to achieve the desired end state of the notebook. 

The A* and path-reconstruction functions were provided in the course homework. I added comments to them, and Claude walked me through a step-by-step trace of how the algorithm runs on our graph. All final design choices and the integration of the code into this notebook were mine. Additionally, to ensure accuracy in reporting my contributions compared to AI written code, I had Claude create a rough draft of this documentation statement. After validation and slight revision, I utilized this statement. 

I worked with ChatGPT to help create correctness checks on our code to ensure that things are working properly.
